# Web Scraping - Lab

## Introduction

Now that you've seen a more extensive example of developing a web scraping script, it's time to further practice and formalize that knowledge by writing functions to parse specific pieces of information from the web page and then synthesizing these into a larger loop that will iterate over successive web pages in order to build a complete dataset.

## Objectives

You will be able to:

* Navigate HTML documents using Beautiful Soup's children and sibling relations
* Select specific elements from HTML using Beautiful Soup
* Use regular expressions to extract items with a certain pattern within Beautiful Soup
* Determine the pagination scheme of a website and scrape multiple pages

## Lab Overview

This lab will build upon the previous lesson. In the end, you'll look to write a script that will iterate over all of the pages for the demo site and extract the title, price, star rating and availability of each book listed. Building up to that, you'll formalize the concepts from the lesson by writing functions that will extract a list of each of these features for each web page. You'll then combine these functions into the full script which will look something like this:  

```python
df = pd.DataFrame()
for i in range(2,51):
    url = "http://books.toscrape.com/catalogue/page-{}.html".format(i)
    soup = BeautifulSoup(html_page.content, 'html.parser')
    new_titles = retrieve_titles(soup)
    new_star_ratings = retrieve_ratings(soup)
    new_prices = retrieve_prices(soup)
    new_avails = retrieve_avails(soup)
    ...
 ```

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [2]:
html_page = requests.get(
    'http://books.toscrape.com/')  # Make a get request to retrieve the page
soup = BeautifulSoup(
    html_page.content,
    'html.parser')  # Pass the page contents to beautiful soup for parsing

## Retrieving Titles

To start, write a function that extracts the titles of the books on a given page. The input for the function should be the `soup` for the HTML of the page.

In [3]:
def retrieve_titles(soup):
    warning = soup.find('div', class_="alert alert-warning")
    book_container = warning.nextSibling.nextSibling
    titles = [
        h3.find('a').attrs['title'] for h3 in book_container.findAll('h3')
    ]
    return titles

In [4]:
retrieve_titles(soup)

['A Light in the Attic',
 'Tipping the Velvet',
 'Soumission',
 'Sharp Objects',
 'Sapiens: A Brief History of Humankind',
 'The Requiem Red',
 'The Dirty Little Secrets of Getting Your Dream Job',
 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics',
 'The Black Maria',
 'Starving Hearts (Triangular Trade Trilogy, #1)',
 "Shakespeare's Sonnets",
 'Set Me Free',
 "Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)",
 'Rip it Up and Start Again',
 'Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991',
 'Olio',
 'Mesaerion: The Best Science Fiction Stories 1800-1849',
 'Libertarianism for Beginners',
 "It's Only the Himalayas"]

## Retrieve Ratings

Next, write a similar function to retrieve the star ratings on a given page. Again, the function should take in the `soup` from the given HTML page and return a list of the star ratings for the books. These star ratings should be formatted as integers.

In [5]:
def retrieve_ratings(soup):
    import re
    warning = soup.find('div', class_="alert alert-warning")
    book_container = warning.nextSibling.nextSibling
    regex = re.compile("star-rating (.*)")
    star_ratings = []
    for p in book_container.findAll('p', {"class": regex}):
        star_ratings.append(p.attrs['class'][-1])
    star_dict = {
        'One': 1,
        'Two': 2,
        'Three': 3,
        'Four': 4,
        'Five': 5
    }  # Manually create a dictionary to translate to numeric
    star_ratings = [star_dict[s] for s in star_ratings]
    return star_ratings

In [6]:
retrieve_ratings(soup)

[3, 1, 1, 4, 5, 1, 4, 3, 4, 1, 2, 4, 5, 5, 5, 3, 1, 1, 2, 2]

## Retrieve Prices

Now write a function to retrieve the prices on a given page. The function should take in the `soup` from the given page and return a list of prices formatted as floats.

In [7]:
def retrieve_prices(soup):
    warning = soup.find('div', class_="alert alert-warning")
    book_container = warning.nextSibling.nextSibling
    prices = [
        float(p.text[1:])
        for p in book_container.findAll('p', class_="price_color")
    ]  # Removing the pound sign and converting to float
    return prices

In [8]:
retrieve_prices(soup)

[51.77,
 53.74,
 50.1,
 47.82,
 54.23,
 22.65,
 33.34,
 17.93,
 22.6,
 52.15,
 13.99,
 20.66,
 17.46,
 52.29,
 35.02,
 57.25,
 23.88,
 37.59,
 51.33,
 45.17]

## Retrieve Availability

Write a function to retrieve whether each book is available or not. The function should take in the `soup` from a given html page and return a list of the availability for each book.

In [9]:
def retrieve_availabilities(soup):
    warning = soup.find('div', class_="alert alert-warning")
    book_container = warning.nextSibling.nextSibling
    avails = [
        a.text.strip()
        for a in book_container.findAll('p', class_="instock availability")
    ]
    return avails

In [10]:
retrieve_availabilities(soup)

['In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock',
 'In stock']

## Create a Script to Retrieve All the Books From All 50 Pages

Finally, write a script to retrieve all of the information from all 50 pages of the books.toscrape.com website. 

In [11]:
df = pd.DataFrame()
for i in range(1, 51):
    url = "http://books.toscrape.com/catalogue/page-{}.html".format(i)
    html_page = requests.get(url)
    soup = BeautifulSoup(html_page.content, 'html.parser')
    warning = soup.find('div', class_="alert alert-warning")
    book_container = warning.nextSibling.nextSibling
    df_1 = pd.DataFrame([
        retrieve_titles(soup),
        retrieve_ratings(soup),
        retrieve_prices(soup),
        retrieve_availabilities(soup)
    ]).transpose()
    df_1.columns = ['Title', 'Star_Rating', 'Price_(pounds)', 'Availability']
    df = df.append(df_1)
df.reset_index().drop(columns=['index'])

,Title,Star_Rating,Price_(pounds),Availability
0,A Light in the Attic,3,51.77,In stock
1,Tipping the Velvet,1,53.74,In stock
2,Soumission,1,50.1,In stock
3,Sharp Objects,4,47.82,In stock
4,Sapiens: A Brief History of Humankind,5,54.23,In stock
...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,1,55.53,In stock
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",4,57.06,In stock
997,A Spy's Devotion (The Regency Spies of London #1),5,16.97,In stock
998,1st to Die (Women's Murder Club #1),1,53.98,In stock


## Level-Up: Write a new version of the script you just wrote. 

If you used URL hacking to generate each successive page URL, instead write a function that retrieves the link from the `"next"` button at the bottom of the page. Conversely, if you already used this approach above, use URL-hacking (arguably the easier of the two methods in this case).

### method 1
> unsuccessfull

In [12]:
def get_next_page(soup):
    next_button = soup.find("li",
                            class_="next")  #May return none if on final page
    if next_button is not None:
        return next_button.find('a').attrs['href']
    else:
        return None

In [13]:
def xy(url):
    df = pd.DataFrame()
    html_page = requests.get(url)
    soup = BeautifulSoup(html_page.content, 'html.parser')

    def xxxx(url):
        warning = soup.find('div', class_="alert alert-warning")
        book_container = warning.nextSibling.nextSibling
        df_1 = pd.DataFrame([
            retrieve_titles(soup),
            retrieve_ratings(soup),
            retrieve_prices(soup),
            retrieve_availabilities(soup)
        ]).transpose()
        df_1.columns = [
            'Title', 'Star_Rating', 'Price_(pounds)', 'Availability'
        ]
        return df_1

    df = df.append(xxxx(url))
#     return df
    next_url_ext = get_next_page(soup)
    while next_url_ext is not None:
        next_url = '/'.join(url.split('/')[:-1]) + '/' + next_url_ext
        print(next_url)
        return df.append(xxxx(next_url))
#         return xxxx(next_url)


In [14]:
xy('http://books.toscrape.com/').reset_index().drop(columns=['index'])

http://books.toscrape.com/catalogue/page-2.html


,Title,Star_Rating,Price_(pounds),Availability
0,A Light in the Attic,3,51.77,In stock
1,Tipping the Velvet,1,53.74,In stock
2,Soumission,1,50.1,In stock
3,Sharp Objects,4,47.82,In stock
4,Sapiens: A Brief History of Humankind,5,54.23,In stock
5,The Requiem Red,1,22.65,In stock
6,The Dirty Little Secrets of Getting Your Dream...,4,33.34,In stock
7,The Coming Woman: A Novel Based on the Life of...,3,17.93,In stock
8,The Boys in the Boat: Nine Americans and Their...,4,22.6,In stock
9,The Black Maria,1,52.15,In stock


### Method 2

#### method 2, raw

In [ ]:
def gnp(soup):
    next_button = soup.find("li",
                            class_="next")  #May return none if on final page
    if next_button:
        return next_button.find('a').attrs['href']
    else:
        return None

In [15]:
def url_list(url,lll=[]):
    html_page = requests.get(url)
    soup = BeautifulSoup(html_page.content, 'html.parser')
    
    lll.append(url)
    next_url_ext = gnp(soup)
    if next_url_ext:
        next_url = '/'.join(url.split('/')[:-1]) + '/' + next_url_ext
        return url_list(next_url)
    else:
        return lll

In [16]:
lennn = url_list('http://books.toscrape.com/')

In [17]:
df = pd.DataFrame()
for i in lennn:
    url = i
    html_page = requests.get(url)
    soup = BeautifulSoup(html_page.content, 'html.parser')
    warning = soup.find('div', class_="alert alert-warning")
    book_container = warning.nextSibling.nextSibling
    df_1 = pd.DataFrame([
        retrieve_titles(soup),
        retrieve_ratings(soup),
        retrieve_prices(soup),
        retrieve_availabilities(soup)
    ]).transpose()
    df_1.columns = ['Title', 'Star_Rating', 'Price_(pounds)', 'Availability']
    df = df.append(df_1)
df.reset_index().drop(columns=['index'])

,Title,Star_Rating,Price_(pounds),Availability
0,A Light in the Attic,3,51.77,In stock
1,Tipping the Velvet,1,53.74,In stock
2,Soumission,1,50.1,In stock
3,Sharp Objects,4,47.82,In stock
4,Sapiens: A Brief History of Humankind,5,54.23,In stock
...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,1,55.53,In stock
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",4,57.06,In stock
997,A Spy's Devotion (The Regency Spies of London #1),5,16.97,In stock
998,1st to Die (Women's Murder Club #1),1,53.98,In stock


#### method 2, func, not working

In [1]:
def get_data(url):
    html_page = requests.get(url)
    soup = BeautifulSoup(html_page.content, 'html.parser')
    
    def gnp(soup):
        next_button = soup.find("li",
                                class_="next")  #May return none if on final page
        if next_button:
            return next_button.find('a').attrs['href']
        else:
            return None
    
    def url_list(url,lll=[]):
        lll.append(url)
        next_url_ext = gnp(soup)
        if next_url_ext:
            next_url = '/'.join(url.split('/')[:-1]) + '/' + next_url_ext
            return url_list(next_url)
        else:
            return lll
    
    links_list = url_list(url)
    df = pd.DataFrame()
    for link in links_list:
        url = link
#         html_page = requests.get(url)
#         soup = BeautifulSoup(html_page.content, 'html.parser')
        warning = soup.find('div', class_="alert alert-warning")
        book_container = warning.nextSibling.nextSibling
        df_1 = pd.DataFrame([
            retrieve_titles(soup),
            retrieve_ratings(soup),
            retrieve_prices(soup),
            retrieve_availabilities(soup)
        ]).transpose()
        df_1.columns = ['Title', 'Star_Rating', 'Price_(pounds)', 'Availability']
        df = df.append(df_1)
        df_c = df.reset_index().drop(columns=['index'])
    return df_c

In [2]:
get_data('http://books.toscrape.com/')

NameError: name 'requests' is not defined

## Summary

Well done! You just completed your first full web scraping project! You're ready to start harnessing the power of the web!